# Wave Glider Preflight — Parse & Cache

**Run this notebook once per dataset** to parse all raw sensor files and save them
to NetCDF in a `cache/` folder inside the mission data directory.

The diagnostic notebook (`preflight_check.ipynb`) loads from this cache — fast.
Re-run this notebook only when raw data changes.

Outputs written to: `<data_path>/cache/`

| File | Contents |
|---|---|
| `rbr.nc` | RBR CTD + CODA oxygen + Cyclops chlorophyll |
| `wxt.nc` | Vaisala WXT met sensors |
| `gill.nc` | Gill sonic anemometer |
| `spn1.nc` | SPN1 shortwave pyranometer |
| `cgr4.nc` | CGR4 longwave pyrgeometer |
| `adcp_beam.nc` | ADCP beam coordinates |
| `adcp_enu.nc` | ADCP ENU coordinates (if present) |
| `imu.nc` | GPSWaves IMU (20 Hz) |
| `gps.nc` | GPSWaves GPS fixes |


---
## 0. Setup

In [1]:
import yaml
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings('ignore')

# --- Load config ---
CONFIG_PATH = Path('config/mission_config.yml')
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

DATA_PATH = Path(cfg['mission']['data_path'])
MISSION   = cfg['mission']['name']
TZ        = cfg['mission'].get('timezone', 'UTC')

tp = cfg.get('test_period', {})
TEST_START  = pd.Timestamp(tp['start'], tz=TZ) if 'start' in tp else None
TEST_END    = pd.Timestamp(tp['end'],   tz=TZ) if 'end'   in tp else None
TEST_PERIOD = (TEST_START, TEST_END) if TEST_START and TEST_END else None

# Sanity filter — rejects obviously corrupt timestamps
# (e.g. year 2113 or 1990 from TOB3 frame errors)
# Set valid_data_range in mission config to match deployment year
vdr = cfg['mission'].get('valid_data_range', {})
GLOBAL_PERIOD = (
    pd.Timestamp(vdr['start'], tz=TZ),
    pd.Timestamp(vdr['end'],   tz=TZ),
) if vdr else None
print(f'Sanity filter: {GLOBAL_PERIOD[0].date()} → {GLOBAL_PERIOD[1].date()}'
      if GLOBAL_PERIOD else 'Sanity filter: none')
print(f'Global filter: {GLOBAL_PERIOD[0]} → {GLOBAL_PERIOD[1]}'
      if GLOBAL_PERIOD else 'Global filter: none')

CACHE_PATH = DATA_PATH / 'cache'
CACHE_PATH.mkdir(exist_ok=True)

print(f'Mission  : {MISSION}')
print(f'Data     : {DATA_PATH}')
print(f'Cache    : {CACHE_PATH}')
print(f'Period   : {TEST_START} → {TEST_END}')


def _strip_tz_from_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Strip timezones from all datetime columns and the index.
    NetCDF4 cannot store tz-aware datetimes — we store UTC and note it in attrs.
    """
    df = df.copy()
    # Drop columns that duplicate the index (e.g. time_gps)
    idx_name = df.index.name
    dup_cols = [c for c in df.columns if c == idx_name or
                (hasattr(df[c], 'dt') and hasattr(df.index, 'tz') and
                 str(df[c].dtype).startswith('datetime') and c == idx_name)]
    if dup_cols:
        df = df.drop(columns=dup_cols)
    # Strip index timezone
    if hasattr(df.index, 'tz') and df.index.tz is not None:
        df.index = df.index.tz_convert('UTC').tz_localize(None)
    # Strip column timezones
    for col in df.columns:
        if hasattr(df[col], 'dt') and hasattr(df[col].dt, 'tz') and df[col].dt.tz is not None:
            df[col] = df[col].dt.tz_convert('UTC').dt.tz_localize(None)
        elif df[col].dtype == object:
            try:
                df[col] = pd.to_datetime(df[col], utc=True).dt.tz_localize(None)
            except Exception:
                df = df.drop(columns=[col])
                print(f'    dropped non-serializable column {col!r}')
    return df


def save(ds, name):
    """Save a DataFrame or xarray Dataset to NetCDF (UTC, tz-naive)."""
    path = CACHE_PATH / f'{name}.nc'
    if isinstance(ds, pd.DataFrame):
        ds_out = _strip_tz_from_df(ds).to_xarray()
        ds_out.attrs['timezone'] = 'UTC'
    else:
        ds_out = ds.copy()
        if 'time' in ds_out.coords:
            t = ds_out.coords['time']
            if str(t.dtype).endswith('UTC]') or 'tz' in str(t.dtype):
                ds_out = ds_out.assign_coords(time=t.values.astype('datetime64[ns]'))
        ds_out.attrs['timezone'] = 'UTC'
    ds_out.to_netcdf(path)
    print(f'  saved → {path.name}  ({path.stat().st_size/1e6:.1f} MB)')
    return ds_out

def load_cache(name, as_dataframe=True):
    """Load a cached NetCDF, restoring UTC timezone on the time index."""
    p = CACHE_PATH / f'{name}.nc'
    if not p.exists():
        print(f'  [cache miss] {name}.nc')
        return None
    ds = xr.open_dataset(p)
    tz_str = ds.attrs.get('timezone', 'UTC')
    if as_dataframe:
        df = ds.to_dataframe()
        # Restore timezone
        if isinstance(df.index, pd.DatetimeIndex):
            df.index = df.index.tz_localize(tz_str)
        return df
    # For xarray: re-attach timezone via pandas round-trip
    if 'time' in ds.coords:
        ds = ds.assign_coords(
            time=pd.DatetimeIndex(ds.time.values).tz_localize(tz_str)
        )
    return ds


Sanity filter: 2026-01-01 → 2026-12-31
Global filter: 2026-01-01 00:00:00+00:00 → 2026-12-31 00:00:00+00:00
Mission  : WHIRLS Mission 3 - Dockside testing
Data     : /Users/xedhjo/Documents/Data/WHIRLS/WHIRLS Mission 3 - Dockside testing
Cache    : /Users/xedhjo/Documents/Data/WHIRLS/WHIRLS Mission 3 - Dockside testing/cache
Period   : None → None


---
## 1. CR6 sensors

Parses all TOB3/TOA5 files for each CR6 sensor table.

In [2]:
from src.loaders import load_cr6

cr6_sensors = ['rbr', 'wxt', 'gill', 'spn1', 'cgr4', 'avg']

for key in cr6_sensors:
    scfg = cfg['cr6'][key]
    if not scfg.get('enabled', True):
        print(f'[SKIP] {scfg["label"]}')
        continue
    print(f'Parsing {scfg["label"]} ...')
    df = load_cr6(
        DATA_PATH, scfg['file_pattern'],
        test_period=GLOBAL_PERIOD, label=scfg['label'],
    )
    if df is not None and not df.empty:
        print(f'  {len(df):,} records  {df.index[0]} → {df.index[-1]}')
        save(df, key)
    else:
        print(f'  [no data]')


Parsing RBR CTD ...
  ⚠️  RBR CTD: 1 record(s) outside test period [2026-01-01 00:00 → 2026-12-31 00:00]
     earliest outlier: 2113-05-09 05:57:29.728000+00:00
     latest outlier:   2113-05-09 05:57:29.728000+00:00
     → excluded from analysis
  33,209 records  2026-05-21 08:11:57+00:00 → 2026-05-26 14:24:32+00:00
  saved → rbr.nc  (3.5 MB)
Parsing Vaisala WXT536 ...
  ⚠️  Vaisala WXT536: 6 record(s) outside test period [2026-01-01 00:00 → 2026-12-31 00:00]
     earliest outlier: 1990-01-01 09:49:10.118400+00:00
     latest outlier:   1990-01-02 23:13:46.598400+00:00
     → excluded from analysis
  33,191 records  2026-05-21 08:12:03+00:00 → 2026-05-26 14:24:20+00:00
  saved → wxt.nc  (2.7 MB)
Parsing Gill Sonic Anemometer ...
  663,565 records  2026-05-21 08:12:05.500000+00:00 → 2026-05-26 14:24:15.400000+00:00
  saved → gill.nc  (42.5 MB)
Parsing SPN1 Pyranometer (shortwave) ...
  1,107 records  2026-05-21 08:00:30+00:00 → 2026-05-26 14:12:30+00:00
  saved → spn1.nc  (0.0 MB)
Pars

---
## 2. ADCP

Parses raw `.ENS` binary. Slow for large files — this is why we cache.

In [3]:
from src.adcp import load_adcp

acfg = cfg['adcp']
if acfg.get('enabled', True):
    print('Parsing ADCP ...')
    adcp_datasets = load_adcp(
        DATA_PATH,
        file_pattern        = acfg['file_pattern'],
        test_period         = TEST_PERIOD,
        sampling_interval_s = int(acfg.get('expected_interval_s', 1)),
    )
    for coord_label, ds in adcp_datasets.items():
        print(f'  {coord_label}: {len(ds.time):,} ensembles')
        save(ds, f'adcp_{coord_label}')
else:
    print('[SKIP] ADCP')


Parsing ADCP ...
Parsing CONFIG1.ENS  (8.0 MB) …
  Ensembles found: 6,320  (ensemble size: 1271 bytes)
  Config: 50 cells × 4 beams, cell size 2.00 m, bin1 3.94 m
  Timestamps: 2026-05-21 11:27:05.790000 → 2026-05-21 13:12:34.790000
  Main segment: 6,320 ensembles
  Coordinate system: beam
  Extracting beam segment (6,320 ensembles) …
    Temperature: 18.28–18.84 °C
    Pressure:    0.04–0.52 dbar
    Heading:     201.7–341.6 °
  beam: 6,320 ensembles
  saved → adcp_beam.nc  (20.7 MB)


---
## 3. GPSWaves + IMU

Parses all `raw.log` files. Can take several minutes for a full day.

In [4]:
from src.loaders import load_gpswaves
from src.loaders import apply_test_period

if cfg['sensors'].get('gpswaves', True):
    print('Parsing GPSWaves / IMU ...')
    df_imu, df_gps = load_gpswaves(DATA_PATH)

    if df_imu is not None and not df_imu.empty:
        df_imu = apply_test_period(df_imu, TEST_PERIOD, label='IMU')
        # Drop non-serializable columns before saving
        drop_cols = [c for c in df_imu.columns
                     if df_imu[c].dtype == object]
        if drop_cols:
            print(f'  dropping object columns: {drop_cols}')
            df_imu = df_imu.drop(columns=drop_cols)
        print(f'  IMU: {len(df_imu):,} records  {df_imu.index[0]} → {df_imu.index[-1]}')
        save(df_imu, 'imu')
    else:
        print('  [no IMU data]')

    if df_gps is not None and not df_gps.empty:
        df_gps = apply_test_period(df_gps, TEST_PERIOD, label='GPS')
        drop_cols = [c for c in df_gps.columns
                     if df_gps[c].dtype == object]
        if drop_cols:
            print(f'  dropping object columns: {drop_cols}')
            df_gps = df_gps.drop(columns=drop_cols)
        print(f'  GPS: {len(df_gps):,} records  {df_gps.index[0]} → {df_gps.index[-1]}')
        save(df_gps, 'gps')
    else:
        print('  [no GPS data]')
else:
    print('[SKIP] GPSWaves')


Parsing GPSWaves / IMU ...
load_gpswaves: found 5 raw.log file(s)
  parsing 20260528235901raw.log … IMU=1179871 GPS=340404 SYSGPS=45544 errors=0
  parsing 20260529235901raw.log … 

KeyboardInterrupt: 

---
## 4. Cache summary

In [5]:
print(f'\nCache contents — {CACHE_PATH}\n')
total = 0
for nc in sorted(CACHE_PATH.glob('*.nc')):
    mb = nc.stat().st_size / 1e6
    total += mb
    ds = xr.open_dataset(nc)
    t_dim = 'time' if 'time' in ds.dims else 'TIMESTAMP' if 'TIMESTAMP' in ds.dims else None
    if t_dim:
        n = len(ds[t_dim])
        t0 = str(ds[t_dim].values[0])[:19]
        t1 = str(ds[t_dim].values[-1])[:19]
        print(f'  {nc.name:<20} {mb:6.1f} MB   {n:>8,} records   {t0} → {t1}')
    else:
        print(f'  {nc.name:<20} {mb:6.1f} MB')
    ds.close()
print(f'\n  Total: {total:.1f} MB')
print('\nDone — run preflight_check.ipynb to continue.')



Cache contents — /Users/xedhjo/Documents/Data/WHIRLS/WHIRLS Mission 3 - Dockside testing/cache

  adcp_beam.nc           20.7 MB      6,320 records   2026-05-21T11:27:05 → 2026-05-21T13:12:34
  avg.nc                  0.0 MB         54 records   2026-05-21T08:10:00 → 2026-05-26T14:00:00
  cgr4.nc                 0.0 MB      1,159 records   2026-05-21T08:00:30 → 2026-05-26T14:23:00
  gill.nc                42.5 MB    663,565 records   2026-05-21T08:12:05 → 2026-05-26T14:24:15
  gps.nc                267.0 MB   1,516,680 records   2026-05-27T23:59:21 → 2026-06-01T09:40:29
  imu.nc               1581.7 MB   7,061,272 records   2026-05-27T23:59:21 → 2026-06-01T09:40:29
  rbr.nc                  3.5 MB     33,209 records   2026-05-21T08:11:57 → 2026-05-26T14:24:32
  spn1.nc                 0.0 MB      1,107 records   2026-05-21T08:00:30 → 2026-05-26T14:12:30
  wxt.nc                  2.7 MB     33,191 records   2026-05-21T08:12:03 → 2026-05-26T14:24:20

  Total: 1918.1 MB

Done — run prefl

---
## 4. Wave statistics & spectra

Parses `waves.log` (bulk Hs/Ta/Tp/Dp) and `spectrum.log` (full directional spectra).
9999 bursts are retained as NaN rows with `valid=False` so gaps are visible.


In [6]:
from src.waves import load_waves, load_spectra

# --- Bulk wave parameters (waves.log) ---
print('Parsing waves.log ...')
df_waves = load_waves(DATA_PATH)
if df_waves is not None and not df_waves.empty:
    n_valid   = df_waves['valid'].sum()
    n_invalid = (~df_waves['valid']).sum()
    print(f'  {len(df_waves):,} bursts total — {n_valid} valid, {n_invalid} invalid (9999)')
    print(f'  {df_waves.index[0]} → {df_waves.index[-1]}')
    if n_valid > 0:
        v = df_waves[df_waves['valid']]
        print(f'  Hs: {v["Hs"].min():.2f}–{v["Hs"].max():.2f} m')
        print(f'  Tp: {v["Tp"].min():.1f}–{v["Tp"].max():.1f} s')
    save(df_waves, 'waves')
else:
    print('  [no waves data]')

# --- Full spectra (spectrum.log) ---
print()
print('Parsing spectrum.log ...')
ds_spectra = load_spectra(DATA_PATH)
if ds_spectra and len(ds_spectra.time) > 0:
    n_valid   = int(ds_spectra['valid'].sum())
    n_invalid = len(ds_spectra.time) - n_valid
    print(f'  {len(ds_spectra.time):,} bursts — {n_valid} valid, {n_invalid} invalid')
    print(f'  {ds_spectra.time.values[0]} → {ds_spectra.time.values[-1]}')
    print(f'  Frequency grid: {float(ds_spectra.frequency[0]):.4f}–'
          f'{float(ds_spectra.frequency[-1]):.4f} Hz  ({len(ds_spectra.frequency)} bins)')
    # Save — strip tz from time coord before writing
    ds_out = ds_spectra.copy()
    ds_out = ds_out.assign_coords(
        time=pd.DatetimeIndex(ds_out.time.values).tz_localize(None)
    )
    ds_out.attrs['timezone'] = 'UTC'
    path_nc = CACHE_PATH / 'spectra.nc'
    ds_out.to_netcdf(path_nc)
    print(f'  saved → spectra.nc  ({path_nc.stat().st_size/1e6:.1f} MB)')
else:
    print('  [no spectra data]')


Parsing waves.log ...
  parsing 20260528235901waves.log … valid=48 invalid/9999=0
  parsing 20260529235901waves.log … valid=47 invalid/9999=1
  parsing 20260530235901waves.log … valid=48 invalid/9999=0
  parsing 20260531235901waves.log … valid=48 invalid/9999=0
  parsing waves.log … valid=16 invalid/9999=4
  212 bursts total — 207 valid, 5 invalid (9999)
  2026-05-27 23:30:00+00:00 → 2026-06-01 09:00:00+00:00
  Hs: 0.12–1.21 m
  Tp: 8.3–17.1 s
  saved → waves.nc  (0.0 MB)

Parsing spectrum.log ...
  parsing 20260528235901spectrum.log … valid=48 invalid/9999=0
  parsing 20260529235901spectrum.log … valid=47 invalid/9999=1
  parsing 20260530235901spectrum.log … valid=48 invalid/9999=0
  parsing 20260531235901spectrum.log … valid=48 invalid/9999=0
  parsing spectrum.log … valid=16 invalid/9999=4
  212 bursts — 207 valid, 5 invalid
  2026-05-27 23:30:00 → 2026-06-01 09:00:00
  Frequency grid: 0.0117–0.3867 Hz  (49 bins)
  saved → spectra.nc  (0.4 MB)


---
## 5. Session detection

Automatically detects contiguous active periods across all CR6 sensors.
Use the output to populate the `sessions` block in `mission_config.yml`.

In [7]:
# =============================================================================
# Session detection — find contiguous active periods across all sensors
# =============================================================================

def detect_sessions(df, sensor_name, min_gap_h=2.0):
    """
    Find contiguous active periods separated by gaps > min_gap_h.
    Returns list of dicts with start, end, n_records, duration_h.
    """
    if df is None or df.empty:
        return []
    dt = df.index.to_series().diff().dt.total_seconds().div(3600).dropna()
    breaks = dt[dt > min_gap_h].index
    sessions, start = [], df.index[0]
    for brk in breaks:
        end = df.index[df.index < brk][-1]
        sessions.append(dict(sensor=sensor_name, start=start, end=end,
                             n=len(df.loc[start:end]),
                             duration_h=round((end-start).total_seconds()/3600, 2)))
        start = df.index[df.index >= brk][0]
    sessions.append(dict(sensor=sensor_name, start=start, end=df.index[-1],
                         n=len(df.loc[start:]),
                         duration_h=round((df.index[-1]-start).total_seconds()/3600, 2)))
    return sessions


MIN_GAP_H = 2.0   # adjust if needed

all_sessions = []
for key in ['rbr', 'wxt', 'gill', 'spn1', 'cgr4']:
    ds = xr.open_dataset(CACHE_PATH / f'{key}.nc')
    df = ds.to_dataframe()
    df.index = pd.DatetimeIndex(df.index).tz_localize('UTC')
    all_sessions.extend(detect_sessions(df, key, MIN_GAP_H))
    ds.close()

df_sess = pd.DataFrame(all_sessions)

# Find consensus sessions — times where most sensors agree
# Round start/end to nearest 30 min to cluster across sensors
df_sess['start_r'] = df_sess['start'].dt.floor('30min')
df_sess['end_r']   = df_sess['end'].dt.ceil('30min')

# Group by rounded start — sensors that start within 30 min = same session
consensus = df_sess.groupby('start_r').agg(
    n_sensors  = ('sensor', 'count'),
    start_min  = ('start', 'min'),
    end_max    = ('end',   'max'),
    sensors    = ('sensor', lambda x: ', '.join(sorted(x))),
).reset_index().sort_values('start_min')

consensus['duration_h'] = (consensus['end_max'] - consensus['start_min']).dt.total_seconds() / 3600
consensus = consensus[consensus['duration_h'] > 0.5]  # drop fragments < 30 min

print('\nDetected sessions (gap threshold: {:.0f}h):\n'.format(MIN_GAP_H))
print(f'{"#":<4} {"Start":<28} {"End":<28} {"Duration":>10}  {"Sensors"}')
print('-' * 90)
for i, row in consensus.iterrows():
    print(f'{i:<4} {str(row.start_min):<28} {str(row.end_max):<28} '
          f'{row.duration_h:>8.1f}h  {row.sensors}')

print('\n--- Suggested config block ---')
print('sessions:')
for i, (_, row) in enumerate(consensus.iterrows(), 1):
    print(f'  - name: "Test {i}"')
    print(f'    start: "{row.start_min.strftime("%Y-%m-%d %H:%M")}"')
    print(f'    end:   "{row.end_max.strftime("%Y-%m-%d %H:%M")}"')
    print(f'    phases: []')
    print(f'    events: []')



Detected sessions (gap threshold: 2h):

#    Start                        End                            Duration  Sensors
------------------------------------------------------------------------------------------
1    2026-05-21 12:11:13.300000+00:00 2026-05-21 15:00:00+00:00         2.8h  cgr4, gill, rbr, spn1, wxt
2    2026-05-26 07:45:00+00:00    2026-05-26 14:23:00+00:00         6.6h  cgr4
3    2026-05-26 08:00:00.050000+00:00 2026-05-26 14:24:32+00:00         6.4h  gill, rbr, spn1, wxt

--- Suggested config block ---
sessions:
  - name: "Test 1"
    start: "2026-05-21 12:11"
    end:   "2026-05-21 15:00"
    phases: []
    events: []
  - name: "Test 2"
    start: "2026-05-26 07:45"
    end:   "2026-05-26 14:23"
    phases: []
    events: []
  - name: "Test 3"
    start: "2026-05-26 08:00"
    end:   "2026-05-26 14:24"
    phases: []
    events: []
